# generator-project-and-reshape — ex1: latent-to-spatial seed projection + reshape

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `generator-project-and-reshape`. Running the final beacon cell reports progress against the `GAN: Generator project + reshape` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Generator project + reshape` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`generator-project-and-reshape`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "generator-project-and-reshape"
DD_SUBTOPIC = "GAN: Generator project + reshape"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Generator project + reshape — quick refresher

The first layer of a DCGAN generator turns a flat noise vector `z (B, 100)` into a spatial seed tensor `(B, 1024, 4, 4)` — the smallest feature map that subsequent ConvTranspose layers will upsample.

```python
self.proj = nn.Linear(latent_dim, 1024 * 4 * 4)  # bias optional
...
x = self.proj(z)                  # (B, 16384)
x = x.view(B, 1024, 4, 4)         # spatial seed
```

**Why `Linear` then `view`, not `ConvTranspose` from a 1×1 seed.** Either works, but `Linear + view` is what the original DCGAN paper (Radford et al.) does — and it's faster (single matmul, no padding math). The result is identical.

**Shape arithmetic.** From `(B, 1024, 4, 4)` four ConvTranspose stride-2 blocks produce `4 -> 8 -> 16 -> 32 -> 64` — final image size 64×64. The channel count halves each time: `1024 -> 512 -> 256 -> 128 -> 3`.

### Exercise 1 — latent-to-spatial seed projection + reshape

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply a `nn.Linear(latent_dim, C*H*W)` followed by `view(B, C, H, W)` to turn a flat noise vector `(B, 100)` into a spatial seed tensor `(B, 1024, 4, 4)` for a DCGAN generator.
> Keywords: gan, generator, linear, view, spatial-seed
> ```

**KCs targeted:** `latent-linear-projection`, `view-to-spatial-seed`

Implement `ex1_project_and_reshape(z, weight, bias, channels, spatial)`. The first layer of every DCGAN generator:

1. `z` has shape `(B, latent_dim)` — typically `latent_dim=100`.
2. `weight` has shape `(channels * spatial * spatial, latent_dim)`, `bias` has shape `(channels * spatial * spatial,)`. Together they parameterize `nn.Linear(latent_dim, channels * spatial * spatial)`.
3. Affine project: `flat = z @ weight.T + bias` → `(B, channels * spatial * spatial)`.
4. Reshape to `(B, channels, spatial, spatial)` with `flat.view(B, channels, spatial, spatial)`.

Input: `z` `(B, latent_dim)`; `weight`/`bias`; `channels` int; `spatial` int.
Output: `(B, channels, spatial, spatial)` float tensor.

The visualization renders the per-channel mean intensity of the first generated seed as a heatmap — useful sanity check that the spatial seed actually varies across channels.

In [ ]:
def ex1_project_and_reshape(z: Tensor, weight: Tensor, bias: Tensor, channels: int, spatial: int) -> Tensor:
    B = z.shape[0]
    flat = z @ weight.T + bias                          # (B, C*H*W)
    return flat.view(B, channels, spatial, spatial)     # (B, C, H, W)


<details><summary>Solution</summary>

```python
def ex1_project_and_reshape(z: Tensor, weight: Tensor, bias: Tensor, channels: int, spatial: int) -> Tensor:
    B = z.shape[0]
    flat = z @ weight.T + bias                          # (B, C*H*W)
    return flat.view(B, channels, spatial, spatial)     # (B, C, H, W)
```

**Why `view`, not `reshape` or `rearrange`.** All three work on a contiguous tensor. `view` is the cheapest (no copy guarantee — throws if not contiguous, which here it always is). `reshape` is more permissive (silently copies if needed). `einops.rearrange(flat, 'b (c h w) -> b c h w', c=channels, h=spatial)` is also fine and more readable.

**Why `Linear + view`, not start from a `(B, 100, 1, 1)` ConvTranspose seed.** Functionally either lets you upsample to `64 × 64`. The DCGAN paper uses Linear + view because (a) it's one matmul vs an expensive transposed convolution from a 1×1 source, and (b) you have direct control over the output channels (`1024 * 4 * 4 = 16384` parameters per output unit, vs the constrained channel-count of a ConvTranspose from 1×1).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()